In [1]:
import torch
print(torch.__version__)

2.7.1+cu118


In [2]:
! mkdir -p ~/work/transformer_chatbot/data/ && cd ~/work/transformer_chatbot/data/
! wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv

--2026-06-22 01:58:41--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-06-22 01:58:42--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv’

ChatbotData.csv     100%[===================>] 868.99K  5.06MB/s    in 0.2s    

2026-06-22 01:58:42 (5.06 MB/s) - ‘ChatbotData.csv’ saved [889842/889842]



In [4]:
import pandas as pd

df = pd.read_csv('~/work/transformer_chatbot/data/ChatbotData.csv')
print(df.shape)
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/work/transformer_chatbot/data/ChatbotData.csv'

In [5]:
import os
import urllib.request
import pandas as pd

os.makedirs('/home/jovyan/work/transformer_chatbot/data/', exist_ok=True)

url = "https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv"
urllib.request.urlretrieve(url, '/home/jovyan/work/transformer_chatbot/data/ChatbotData.csv')

df = pd.read_csv('/home/jovyan/work/transformer_chatbot/data/ChatbotData.csv')
print(df.shape)
print(df.head())

(11823, 3)
                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0


In [6]:
import re

# 결측값 제거
df = df.dropna()
print(f"결측값 제거 후: {df.shape}")

# 특수문자 제거 함수
def preprocess(text):
    text = re.sub(r"[^가-힣a-zA-Z0-9?.!,\s]", "", text)  # 한글, 영문, 숫자, 기본 구두점만 유지
    text = re.sub(r"\s+", " ", text).strip()               # 중복 공백 제거
    return text

df['Q'] = df['Q'].apply(preprocess)
df['A'] = df['A'].apply(preprocess)

# 빈 문장 제거
df = df[(df['Q'] != '') & (df['A'] != '')]
print(f"전처리 후: {df.shape}")
print(df.head())

결측값 제거 후: (11823, 3)
전처리 후: (11823, 3)
                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0


In [7]:
import sentencepiece as spm
import pandas as pd
import os

# SentencePiece 학습용 텍스트 파일 생성 (Q, A 합쳐서)
data_path = '/home/jovyan/work/transformer_chatbot/data/'
txt_path = data_path + 'chatbot_corpus.txt'

with open(txt_path, 'w', encoding='utf-8') as f:
    for text in pd.concat([df['Q'], df['A']]):
        f.write(text.strip() + '\n')

print(f"코퍼스 문장 수: {len(df['Q']) + len(df['A'])}")

# SentencePiece 모델 학습
spm.SentencePieceTrainer.train(
    input=txt_path,
    model_prefix=data_path + 'chatbot_spm',
    vocab_size=8000,
    character_coverage=0.9995,  # 한국어는 0.9995 권장
    model_type='bpe',
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece='[PAD]',
    unk_piece='[UNK]',
    bos_piece='[BOS]',
    eos_piece='[EOS]',
)

# 학습된 모델 로드
sp = spm.SentencePieceProcessor()
sp.load(data_path + 'chatbot_spm.model')

# 테스트
test = "여행은 언제나 좋죠."
print(f"\n원문: {test}")
print(f"토크나이징: {sp.encode_as_pieces(test)}")
print(f"ID 변환: {sp.encode_as_ids(test)}")

코퍼스 문장 수: 23646

원문: 여행은 언제나 좋죠.
토크나이징: ['▁여행은', '▁언제나', '▁좋죠', '.']
ID 변환: [5128, 1356, 379, 6926]


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /home/jovyan/work/transformer_chatbot/data/chatbot_corpus.txt
  input_format: 
  model_prefix: /home/jovyan/work/transformer_chatbot/data/chatbot_spm
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: [UNK]
  bos_piece: [BOS]
  eo

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

# 하이퍼파라미터
VOCAB_SIZE = 8000
MAX_LEN = 40
BATCH_SIZE = 64
PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

# Q, A 토크나이징
def encode(text, sp, max_len):
    ids = [BOS_ID] + sp.encode_as_ids(text) + [EOS_ID]
    # max_len에 맞게 자르거나 패딩
    ids = ids[:max_len]
    ids += [PAD_ID] * (max_len - len(ids))
    return ids

# 데이터셋 클래스
class ChatDataset(Dataset):
    def __init__(self, df, sp, max_len):
        self.questions = [encode(q, sp, max_len) for q in df['Q']]
        self.answers   = [encode(a, sp, max_len) for a in df['A']]

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.questions[idx], dtype=torch.long),
            torch.tensor(self.answers[idx],   dtype=torch.long),
        )

# 학습/검증 분리 (9:1)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = ChatDataset(train_df, sp, MAX_LEN)
val_dataset   = ChatDataset(val_df,   sp, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

print(f"학습 데이터: {len(train_dataset)}개")
print(f"검증 데이터: {len(val_dataset)}개")

# 샘플 확인
q, a = train_dataset[0]
print(f"\n질문 텐서: {q}")
print(f"답변 텐서: {a}")

학습 데이터: 10640개
검증 데이터: 1183개

질문 텐서: tensor([   2,  543,  563, 1784, 2025, 2115,    3,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])
답변 텐서: tensor([   2,  797, 6785, 3601, 6928,  338, 2448, 5603, 6993, 2821, 3970, 6926,
           3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])


In [10]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # (max_len, d_model) 크기의 PE 행렬 생성
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # 짝수 인덱스 → sin, 홀수 인덱스 → cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model) → 배치 브로드캐스팅용
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# 테스트
pe = PositionalEncoding(d_model=128)
x = torch.zeros(2, 10, 128)  # (batch=2, seq_len=10, d_model=128)
out = pe(x)
print("PositionalEncoding 출력 shape:", out.shape)  # (2, 10, 128)

PositionalEncoding 출력 shape: torch.Size([2, 10, 128])


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scaled_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)

    attention_weights = F.softmax(scaled_scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads

        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense   = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)
        self.out_dense   = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        query = self.split_heads(self.query_dense(query), batch_size)
        key   = self.split_heads(self.key_dense(key),     batch_size)
        value = self.split_heads(self.value_dense(value), batch_size)

        scaled_attention, _ = scaled_dot_product_attention(query, key, value, mask)
        scaled_attention = scaled_attention.permute(0, 2, 1, 3).contiguous()
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)

        return self.out_dense(concat_attention)

In [13]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()

        # Multi-Head Attention
        self.attention = MultiHeadAttention(d_model, num_heads)

        # Feed Forward Network
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

        # Layer Normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1. Multi-Head Attention + Add & Norm
        attn_out = self.attention(x, x, x, mask)        # Self-Attention (Q=K=V=x)
        x = self.norm1(x + self.dropout(attn_out))      # Residual Connection

        # 2. Feed Forward + Add & Norm
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))        # Residual Connection

        return x

# 테스트
encoder_layer = EncoderLayer(d_model=128, num_heads=8, d_ff=256)
x = torch.zeros(2, 10, 128)  # (batch=2, seq_len=10, d_model=128)
out = encoder_layer(x)
print("EncoderLayer 출력 shape:", out.shape)  # (2, 10, 128)

EncoderLayer 출력 shape: torch.Size([2, 10, 128])


In [14]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()

        # 1. Masked Multi-Head Attention (Self-Attention)
        self.attention1 = MultiHeadAttention(d_model, num_heads)

        # 2. Cross Attention (Encoder 출력을 K, V로 사용)
        self.attention2 = MultiHeadAttention(d_model, num_heads)

        # 3. Feed Forward Network
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

        # Layer Normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, look_ahead_mask=None, padding_mask=None):
        # 1. Masked Self-Attention + Add & Norm
        # 미래 토큰을 보지 못하도록 look_ahead_mask 적용
        attn1 = self.attention1(x, x, x, look_ahead_mask)
        x = self.norm1(x + self.dropout(attn1))

        # 2. Cross Attention + Add & Norm
        # Q: 디코더 출력, K/V: 인코더 출력
        attn2 = self.attention2(x, enc_output, enc_output, padding_mask)
        x = self.norm2(x + self.dropout(attn2))

        # 3. Feed Forward + Add & Norm
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))

        return x

# 테스트
decoder_layer = DecoderLayer(d_model=128, num_heads=8, d_ff=256)
x          = torch.zeros(2, 10, 128)  # 디코더 입력
enc_output = torch.zeros(2, 10, 128)  # 인코더 출력
out = decoder_layer(x, enc_output)
print("DecoderLayer 출력 shape:", out.shape)  # (2, 10, 128)

DecoderLayer 출력 shape: torch.Size([2, 10, 128])


In [15]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len, dropout=0.1):
        super(Transformer, self).__init__()

        # 임베딩 (인코더/디코더 공유)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

        # 인코더/디코더 레이어 스택
        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.decoder_layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

        # 최종 출력 레이어
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def encode(self, src, src_mask=None):
        # src: (batch_size, src_len)
        x = self.dropout(self.pos_encoding(self.embedding(src)))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, enc_output, look_ahead_mask=None, padding_mask=None):
        # tgt: (batch_size, tgt_len)
        x = self.dropout(self.pos_encoding(self.embedding(tgt)))
        for layer in self.decoder_layers:
            x = layer(x, enc_output, look_ahead_mask, padding_mask)
        return x

    def forward(self, src, tgt, src_mask=None, look_ahead_mask=None):
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, look_ahead_mask, src_mask)
        return self.fc_out(dec_output)  # (batch_size, tgt_len, vocab_size)


# 하이퍼파라미터
D_MODEL    = 128
NUM_HEADS  = 8
D_FF       = 256
NUM_LAYERS = 2
DROPOUT    = 0.1

model = Transformer(
    vocab_size  = VOCAB_SIZE,
    d_model     = D_MODEL,
    num_heads   = NUM_HEADS,
    d_ff        = D_FF,
    num_layers  = NUM_LAYERS,
    max_len     = MAX_LEN,
    dropout     = DROPOUT
)

# 테스트
src = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN))
tgt = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN))
out = model(src, tgt)
print("Transformer 출력 shape:", out.shape)  # (2, MAX_LEN, VOCAB_SIZE)
print(f"파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Transformer 출력 shape: torch.Size([2, 40, 8000])
파라미터 수: 2,718,528


In [17]:
import torch.optim as optim

device = torch.device('cuda')

model = model.to(device)

# 손실 함수 (PAD 토큰은 loss 계산에서 제외)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def create_padding_mask(x):
    # 패딩 위치는 0, 나머지는 1
    mask = (x != PAD_ID).float()
    return mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)

def create_look_ahead_mask(x):
    seq_len = x.size(1)
    look_ahead = 1 - torch.tril(torch.ones(seq_len, seq_len, device=x.device))
    look_ahead = look_ahead.unsqueeze(0).unsqueeze(1)  # (1, 1, seq_len, seq_len)
    padding = create_padding_mask(x)                   # (batch, 1, 1, seq_len)
    combined = torch.max(look_ahead, 1 - padding)
    return combined  # (batch, 1, seq_len, seq_len)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)

        # 디코더 입력: [BOS, ...] / 디코더 타깃: [..., EOS]
        tgt_input  = tgt[:, :-1]
        tgt_target = tgt[:, 1:]

        # 마스크 생성
        src_mask         = create_padding_mask(src)
        look_ahead_mask  = create_look_ahead_mask(tgt_input)

        optimizer.zero_grad()
        output = model(src, tgt_input, src_mask, look_ahead_mask)
        # output: (batch, tgt_len-1, vocab_size)

        # loss 계산
        output = output.reshape(-1, VOCAB_SIZE)
        tgt_target = tgt_target.reshape(-1)
        loss = criterion(output, tgt_target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)

            tgt_input  = tgt[:, :-1]
            tgt_target = tgt[:, 1:]

            src_mask        = create_padding_mask(src)
            look_ahead_mask = create_look_ahead_mask(tgt_input)

            output = model(src, tgt_input, src_mask, look_ahead_mask)
            output = output.reshape(-1, VOCAB_SIZE)
            tgt_target = tgt_target.reshape(-1)

            loss = criterion(output, tgt_target)
            total_loss += loss.item()

    return total_loss / len(loader)

# 학습 실행
EPOCHS = 20
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = eval_epoch(model, val_loader, criterion)

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # 최적 모델 저장
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/home/jovyan/work/transformer_chatbot/best_model.pt')
        print(f"          → 모델 저장 (Val Loss: {val_loss:.4f})")

Epoch 01 | Train Loss: 0.0274 | Val Loss: 0.1232
          → 모델 저장 (Val Loss: 0.1232)
Epoch 02 | Train Loss: 0.0154 | Val Loss: 0.1361
Epoch 03 | Train Loss: 0.0124 | Val Loss: 0.1401
Epoch 04 | Train Loss: 0.0138 | Val Loss: 0.1469
Epoch 05 | Train Loss: 0.0123 | Val Loss: 0.1493
Epoch 06 | Train Loss: 0.0138 | Val Loss: 0.1507
Epoch 07 | Train Loss: 0.0113 | Val Loss: 0.1492
Epoch 08 | Train Loss: 0.0092 | Val Loss: 0.1553
Epoch 09 | Train Loss: 0.0074 | Val Loss: 0.1493
Epoch 10 | Train Loss: 0.0085 | Val Loss: 0.1569
Epoch 11 | Train Loss: 0.0083 | Val Loss: 0.1582
Epoch 12 | Train Loss: 0.0092 | Val Loss: 0.1598
Epoch 13 | Train Loss: 0.0089 | Val Loss: 0.1619
Epoch 14 | Train Loss: 0.0072 | Val Loss: 0.1604
Epoch 15 | Train Loss: 0.0093 | Val Loss: 0.1602
Epoch 16 | Train Loss: 0.0093 | Val Loss: 0.1730
Epoch 17 | Train Loss: 0.0101 | Val Loss: 0.1719
Epoch 18 | Train Loss: 0.0082 | Val Loss: 0.1671
Epoch 19 | Train Loss: 0.0078 | Val Loss: 0.1714
Epoch 20 | Train Loss: 0.0065 | 

In [18]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 새로 초기화 (과적합 방지를 위해 dropout 높임)
# 문제: dropout=0.1로 너무 낮아서 모델이 학습 데이터를 그대로 외워버림
model = Transformer(
    vocab_size  = VOCAB_SIZE,
    d_model     = D_MODEL,
    num_heads   = NUM_HEADS,
    d_ff        = D_FF,
    num_layers  = NUM_LAYERS,
    max_len     = MAX_LEN,
    dropout     = 0.3  # 0.1 → 0.3으로 높여서 과적합 억제
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# 문제: lr=1e-3이 너무 높아서 Epoch 1 이후 val loss가 튀어오름
# 학습률을 낮추고 스케줄러로 점진적으로 감소시킴
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,      # val loss 개선 없으면 lr을 절반으로 줄임
    patience=3,      # 3 epoch 동안 개선 없으면 lr 감소
)

EPOCHS    = 50
best_val_loss   = float('inf')
early_stop_count = 0
EARLY_STOP_PATIENCE = 7  # 7 epoch 동안 개선 없으면 학습 중단

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = eval_epoch(model, val_loader, criterion)

    scheduler.step(val_loss)  # val loss 기준으로 lr 조정
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        early_stop_count = 0
        torch.save(model.state_dict(), '/home/jovyan/work/transformer_chatbot/best_model.pt')
        print(f"          → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        # 문제: 개선 없이 계속 학습하면 과적합만 심해짐
        # early stopping으로 최적 시점에서 학습 중단
        if early_stop_count >= EARLY_STOP_PATIENCE:
            print(f"\n{EARLY_STOP_PATIENCE} epoch 동안 개선 없음 → 학습 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

Epoch 01 | Train Loss: 6.4877 | Val Loss: 5.6481 | LR: 0.000300
          → 모델 저장 (Val Loss: 5.6481)
Epoch 02 | Train Loss: 5.5281 | Val Loss: 5.4244 | LR: 0.000300
          → 모델 저장 (Val Loss: 5.4244)
Epoch 03 | Train Loss: 5.2830 | Val Loss: 5.1220 | LR: 0.000300
          → 모델 저장 (Val Loss: 5.1220)
Epoch 04 | Train Loss: 4.9930 | Val Loss: 4.7365 | LR: 0.000300
          → 모델 저장 (Val Loss: 4.7365)
Epoch 05 | Train Loss: 4.6785 | Val Loss: 4.3047 | LR: 0.000300
          → 모델 저장 (Val Loss: 4.3047)
Epoch 06 | Train Loss: 4.3599 | Val Loss: 3.8944 | LR: 0.000300
          → 모델 저장 (Val Loss: 3.8944)
Epoch 07 | Train Loss: 4.0561 | Val Loss: 3.4632 | LR: 0.000300
          → 모델 저장 (Val Loss: 3.4632)
Epoch 08 | Train Loss: 3.7686 | Val Loss: 3.0975 | LR: 0.000300
          → 모델 저장 (Val Loss: 3.0975)
Epoch 09 | Train Loss: 3.5030 | Val Loss: 2.7710 | LR: 0.000300
          → 모델 저장 (Val Loss: 2.7710)
Epoch 10 | Train Loss: 3.2617 | Val Loss: 2.4794 | LR: 0.000300
          → 모델 저장 (Val Loss

In [19]:
def predict(sentence, model, sp, max_len=MAX_LEN):
    model.eval()

    # 입력 문장 전처리 및 토크나이징
    sentence = preprocess(sentence)
    src_ids = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    src_ids += [PAD_ID] * (max_len - len(src_ids))
    src = torch.tensor([src_ids], dtype=torch.long).to(device)  # (1, max_len)

    # 인코더 출력 계산 (한 번만 실행)
    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_output = model.encode(src, src_mask)

    # 디코더 입력 초기화 (BOS 토큰으로 시작)
    tgt_ids = [BOS_ID]

    for _ in range(max_len):
        tgt = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_output = model.decode(tgt, enc_output, look_ahead_mask, src_mask)
            # 마지막 토큰의 예측값으로 다음 토큰 선택
            next_token = dec_output[:, -1, :]          # (1, vocab_size)
            next_token = model.fc_out(next_token)
            next_id = next_token.argmax(dim=-1).item() # 가장 높은 확률의 토큰

        # EOS 토큰이 나오면 생성 종료
        if next_id == EOS_ID:
            break

        tgt_ids.append(next_id)

    # BOS 제외하고 디코딩
    answer = sp.decode_ids(tgt_ids[1:])
    return answer

# 테스트
print(predict("오늘 기분이 너무 안좋아", model, sp))
print(predict("나 요즘 너무 외로워", model, sp))
print(predict("밥은 먹었어?", model, sp))

하아 하아 하아 하아 하아 하아 하아 골 하아 골 하아 골 하아 골 하아 골 하아 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골
하아 하아 하아 하아 하아 하아 하아 골 하아 골 하아 골 하아 골 하아 골 하아 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골 골
피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부 피부


In [20]:
def predict(sentence, model, sp, max_len=MAX_LEN, temperature=0.7, repetition_penalty=1.3):
    model.eval()

    sentence = preprocess(sentence)
    src_ids = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    src_ids += [PAD_ID] * (max_len - len(src_ids))
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_output = model.encode(src, src_mask)

    tgt_ids = [BOS_ID]

    for _ in range(max_len):
        tgt = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_output = model.decode(tgt, enc_output, look_ahead_mask, src_mask)
            logits = model.fc_out(dec_output[:, -1, :])  # (1, vocab_size)

            # 반복 패널티: 이미 생성한 토큰의 확률을 낮춤
            for token_id in set(tgt_ids):
                logits[0, token_id] /= repetition_penalty

            # Temperature: 값이 낮을수록 더 확실한 토큰 선택
            logits = logits / temperature

            # 확률 분포에서 샘플링 (argmax 대신 다양성 확보)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == EOS_ID:
            break

        tgt_ids.append(next_id)

    return sp.decode_ids(tgt_ids[1:])

# 테스트
print(predict("오늘 기분이 너무 안좋아", model, sp))
print(predict("나 요즘 너무 외로워", model, sp))
print(predict("밥은 먹었어?", model, sp))

피부 아이스크림 응원합니다 육아 호랑이보다 깨요 당연 응원합니다 탁 음료 응원합니다
북 응원합니다 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 재밌 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림
응원합니다 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 재밌 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림 아이스크림


In [21]:
def predict(sentence, model, sp, max_len=MAX_LEN, temperature=0.7, repetition_penalty=2.5, top_k=10):
    model.eval()

    sentence = preprocess(sentence)
    src_ids = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    src_ids += [PAD_ID] * (max_len - len(src_ids))
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_output = model.encode(src, src_mask)

    tgt_ids = [BOS_ID]

    for _ in range(max_len):
        tgt = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_output = model.decode(tgt, enc_output, look_ahead_mask, src_mask)
            logits = model.fc_out(dec_output[:, -1, :])  # (1, vocab_size)

            # 반복 패널티 강화: 이미 나온 토큰 확률을 강하게 낮춤
            for token_id in set(tgt_ids):
                logits[0, token_id] /= repetition_penalty

            # PAD, BOS, UNK는 생성 불가하도록 완전 차단
            logits[0, PAD_ID] = -1e9
            logits[0, BOS_ID] = -1e9
            logits[0, 1]      = -1e9  # UNK_ID

            logits = logits / temperature

            # Top-K 샘플링: 상위 K개 토큰 중에서만 선택 (나머지는 차단)
            top_k_logits, top_k_ids = torch.topk(logits, top_k, dim=-1)
            probs = torch.softmax(top_k_logits, dim=-1)
            next_id = top_k_ids[0, torch.multinomial(probs, num_samples=1).item()].item()

        if next_id == EOS_ID:
            break

        tgt_ids.append(next_id)

    return sp.decode_ids(tgt_ids[1:])

# 테스트
print(predict("오늘 기분이 너무 안좋아", model, sp))
print(predict("나 요즘 너무 외로워", model, sp))
print(predict("밥은 먹었어?", model, sp))

피부 부자 과식하지 응원합니다 아이스크림 망 재밌 짜장면 바다 이야기해 크게니 디 보고제를맥 졸업빛
아이스크림 응원합니다 짜장면 분이 크리스마스 샌드위치 재밌하던릿빛여
아이스크림 시원한 응원합니다 네 과식하지 하아 골 사귀기 전에 재밌하던 자연스럽게 사람하고 지금이라도 효 것도 이왕이면 만드는빛여 혼자 여행도 지


In [23]:
# 모델 크기 키우기 (더 많은 파라미터 = 더 풍부한 표현력)
D_MODEL    = 256   # 128 → 256
NUM_HEADS  = 8
D_FF       = 512   # 256 → 512
NUM_LAYERS = 3     # 2 → 3
DROPOUT    = 0.2   # 0.3 → 0.2 (모델이 커졌으니 dropout 살짝 줄임)

model = Transformer(
    vocab_size  = VOCAB_SIZE,
    d_model     = D_MODEL,
    num_heads   = NUM_HEADS,
    d_ff        = D_FF,
    num_layers  = NUM_LAYERS,
    max_len     = MAX_LEN,
    dropout     = DROPOUT
).to(device)

print(f"파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

EPOCHS = 100              # 50 → 100으로 늘림
best_val_loss    = float('inf')
early_stop_count = 0
EARLY_STOP_PATIENCE = 10  # 7 → 10 (더 여유있게 기다림)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = eval_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        early_stop_count = 0
        torch.save(model.state_dict(), '/home/jovyan/work/transformer_chatbot/best_model.pt')
        print(f"          → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        # early stopping: 10 epoch 동안 val loss 개선 없으면 중단
        if early_stop_count >= EARLY_STOP_PATIENCE:
            print(f"\n{EARLY_STOP_PATIENCE} epoch 동안 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

파라미터 수: 8,057,664
Epoch 01 | Train Loss: 5.7968 | Val Loss: 5.0372 | LR: 0.000300
          → 모델 저장 (Val Loss: 5.0372)
Epoch 02 | Train Loss: 4.5286 | Val Loss: 3.7581 | LR: 0.000300
          → 모델 저장 (Val Loss: 3.7581)
Epoch 03 | Train Loss: 3.5345 | Val Loss: 2.7925 | LR: 0.000300
          → 모델 저장 (Val Loss: 2.7925)
Epoch 04 | Train Loss: 2.7679 | Val Loss: 2.0639 | LR: 0.000300
          → 모델 저장 (Val Loss: 2.0639)
Epoch 05 | Train Loss: 2.1501 | Val Loss: 1.5163 | LR: 0.000300
          → 모델 저장 (Val Loss: 1.5163)
Epoch 06 | Train Loss: 1.6421 | Val Loss: 1.0914 | LR: 0.000300
          → 모델 저장 (Val Loss: 1.0914)
Epoch 07 | Train Loss: 1.2280 | Val Loss: 0.7813 | LR: 0.000300
          → 모델 저장 (Val Loss: 0.7813)
Epoch 08 | Train Loss: 0.8883 | Val Loss: 0.5541 | LR: 0.000300
          → 모델 저장 (Val Loss: 0.5541)
Epoch 09 | Train Loss: 0.6313 | Val Loss: 0.3984 | LR: 0.000300
          → 모델 저장 (Val Loss: 0.3984)
Epoch 10 | Train Loss: 0.4411 | Val Loss: 0.3023 | LR: 0.000300
         

In [24]:
# 저장된 최적 모델 불러오기
model.load_state_dict(torch.load('/home/jovyan/work/transformer_chatbot/best_model.pt'))
print("모델 로드 완료")

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict(t, model, sp)}")
    print()

모델 로드 완료
Q: 오늘 기분이 너무 안좋아
A: 

Q: 나 요즘 너무 외로워
A: 진실

Q: 밥은 먹었어?
A: 

Q: 취업이 너무 힘들어
A: 

Q: 사랑이 뭔지 모르겠어
A: 



In [25]:
def predict(sentence, model, sp, max_len=MAX_LEN,
            temperature=0.5,         # 너무 낮으면 EOS만 계속 뽑힘, 0.3→0.5로 올림
            repetition_penalty=2.0,  # 3.0→2.0으로 줄임 (너무 강하면 EOS로 도망감)
            top_k=10,                # 5→10으로 늘림 (후보 토큰 더 다양하게)
            min_len=5):              # 최소 5토큰 생성 전까지 EOS 차단
    model.eval()

    sentence = preprocess(sentence)
    src_ids = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    src_ids += [PAD_ID] * (max_len - len(src_ids))
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_output = model.encode(src, src_mask)

    tgt_ids  = [BOS_ID]
    recent_window = 4  # 최근 4개 토큰은 다시 못 뽑게 함

    for step in range(max_len):
        tgt = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_output = model.decode(tgt, enc_output, look_ahead_mask, src_mask)
            logits = model.fc_out(dec_output[:, -1, :])

            # 특수 토큰 차단
            logits[0, PAD_ID] = -1e9
            logits[0, BOS_ID] = -1e9
            logits[0, 1]      = -1e9  # UNK

            # 최소 길이 전까지 EOS 차단 (너무 일찍 끝나는 문제 방지)
            if step < min_len:
                logits[0, EOS_ID] = -1e9

            # 전체 생성 토큰 반복 패널티
            for token_id in set(tgt_ids):
                logits[0, token_id] /= repetition_penalty

            # 최근 window 토큰 완전 차단
            for token_id in tgt_ids[-recent_window:]:
                logits[0, token_id] = -1e9

            logits = logits / temperature

            # Top-K 샘플링
            top_k_logits, top_k_ids = torch.topk(logits, top_k, dim=-1)
            probs   = torch.softmax(top_k_logits, dim=-1)
            next_id = top_k_ids[0, torch.multinomial(probs, num_samples=1).item()].item()

        if next_id == EOS_ID:
            break

        tgt_ids.append(next_id)

    return sp.decode_ids(tgt_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict(t, model, sp)}")
    print()

Q: 오늘 기분이 너무 안좋아
A: 진실 상대방이 감정에 응원합니다!

Q: 나 요즘 너무 외로워
A: 진실 충분히 내일은 상대방이!

Q: 밥은 먹었어?
A: 감기 감정에 진실 마스크 사랑의

Q: 취업이 너무 힘들어
A: 상대방이! 축하해요 응원합니다 피부

Q: 사랑이 뭔지 모르겠어
A: 감기 조심 마스크면서 감정에 약 때



In [26]:
def predict_beam(sentence, model, sp, max_len=MAX_LEN,
                 beam_size=5,    # 동시에 탐색할 후보 수
                 min_len=3):     # 최소 생성 길이
    model.eval()

    sentence = preprocess(sentence)
    src_ids = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    src_ids += [PAD_ID] * (max_len - len(src_ids))
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_output = model.encode(src, src_mask)

    # beam: (누적 log 확률, 생성된 토큰 리스트)
    beams = [(0.0, [BOS_ID])]
    completed = []  # EOS까지 완성된 후보들

    for step in range(max_len):
        new_beams = []

        for score, tgt_ids in beams:
            tgt = torch.tensor([tgt_ids], dtype=torch.long).to(device)
            look_ahead_mask = create_look_ahead_mask(tgt)

            with torch.no_grad():
                dec_output = model.decode(tgt, enc_output, look_ahead_mask, src_mask)
                logits = model.fc_out(dec_output[:, -1, :])

                # 특수 토큰 차단
                logits[0, PAD_ID] = -1e9
                logits[0, BOS_ID] = -1e9
                logits[0, 1]      = -1e9  # UNK

                # 최소 길이 전까지 EOS 차단
                if step < min_len:
                    logits[0, EOS_ID] = -1e9

                log_probs = torch.log_softmax(logits, dim=-1)

                # 상위 beam_size개 토큰 후보 선택
                top_log_probs, top_ids = torch.topk(log_probs, beam_size, dim=-1)

            for i in range(beam_size):
                next_id    = top_ids[0, i].item()
                next_score = score + top_log_probs[0, i].item()

                if next_id == EOS_ID:
                    # 길이로 점수 정규화 (짧은 문장이 유리한 문제 방지)
                    normalized = next_score / len(tgt_ids)
                    completed.append((normalized, tgt_ids[1:]))
                else:
                    new_beams.append((next_score, tgt_ids + [next_id]))

        if not new_beams:
            break

        # 점수 기준 상위 beam_size개만 유지
        new_beams.sort(key=lambda x: x[0], reverse=True)
        beams = new_beams[:beam_size]

        if len(completed) >= beam_size:
            break

    # 완성된 후보 없으면 현재 beams 중 최고 사용
    if not completed:
        completed = [(s / len(ids), ids[1:]) for s, ids in beams]

    # 점수 가장 높은 후보 선택
    completed.sort(key=lambda x: x[0], reverse=True)
    best_ids = completed[0][1]

    return sp.decode_ids(best_ids)

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict_beam(t, model, sp)}")
    print()

Q: 오늘 기분이 너무 안좋아
A: 진실 진실 진실 진실

Q: 나 요즘 너무 외로워
A: 진실 진실 진실 진실

Q: 밥은 먹었어?
A: 진실 진실 진실 진실

Q: 취업이 너무 힘들어
A: 진실 진실 진실 진실

Q: 사랑이 뭔지 모르겠어
A: 진실 진실 진실 진실



In [27]:
# "진실"이 데이터셋에 얼마나 많이 나오는지 확인
# 모델이 "진실"만 뽑는 건 이 단어가 답변에 너무 많이 등장하기 때문일 수 있음
truth_count = df['A'].str.contains('진실').sum()
print(f"'진실' 포함 답변 수: {truth_count}")

# 가장 많이 등장하는 답변 상위 20개 확인
# 특정 답변이 너무 많으면 모델이 그것만 학습함
print("\n가장 많이 등장하는 답변 TOP 20:")
print(df['A'].value_counts().head(20))

'진실' 포함 답변 수: 8

가장 많이 등장하는 답변 TOP 20:
A
맛있게 드세요.                               22
제가 있잖아요.                               17
조심하세요.                                 14
감기 조심하세요.                              14
직접 물어보세요.                              12
잘할 수 있을 거예요.                           12
많은 시간이 흘렀네요.                           10
맘고생 많았어요.                              10
맛있는 거 드세요.                             10
지금도 늦지 않았어요.                           10
자연스러운 현상이에요.                           10
네 말씀해주세요.                              10
로맨틱하네요.                                 9
금방 지나갈 거예요.                             9
호감이 있을 수도 있어요. 그렇지만 조금 더 상황을 지켜보세요.     9
잘 찾아보세요.                                9
힘내세요.                                   9
저랑 놀아요.                                 9
그럴 수 있어요.                               8
회원정보 찾기를 해보세요.                          8
Name: count, dtype: int64


In [28]:
# "진실" 토큰 ID 확인
print("'진실' 토큰 ID:", sp.encode_as_ids('진실'))
print("'진실' 토크나이징:", sp.encode_as_pieces('진실'))

# 모델이 가장 자주 예측하는 토큰 TOP 20 확인
# → 모델이 실제로 어떤 토큰에 편향되어 있는지 파악
model.eval()
token_counts = torch.zeros(VOCAB_SIZE)

with torch.no_grad():
    for src, tgt in val_loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_input  = tgt[:, :-1]
        src_mask        = create_padding_mask(src)
        look_ahead_mask = create_look_ahead_mask(tgt_input)

        output = model(src, tgt_input, src_mask, look_ahead_mask)
        preds  = output.argmax(dim=-1).cpu().flatten()

        for p in preds:
            token_counts[p] += 1

# 가장 많이 예측된 토큰 TOP 20
top_tokens = token_counts.argsort(descending=True)[:20]
print("\n모델이 가장 많이 예측하는 토큰 TOP 20:")
for tid in top_tokens:
    tid = tid.item()
    print(f"  ID {tid:5d} | '{sp.id_to_piece(tid)}' | 예측 횟수: {int(token_counts[tid])}")

'진실' 토큰 ID: [2588]
'진실' 토크나이징: ['▁진실']

모델이 가장 많이 예측하는 토큰 TOP 20:
  ID     3 | '[EOS]' | 예측 횟수: 39335
  ID  6926 | '.' | 예측 횟수: 1169
  ID    21 | '▁거예요' | 예측 횟수: 93
  ID    62 | '▁더' | 예측 횟수: 56
  ID    29 | '▁수' | 예측 횟수: 46
  ID   112 | '▁있어요' | 예측 횟수: 42
  ID   108 | '▁같아요' | 예측 횟수: 42
  ID    80 | '▁좋은' | 예측 횟수: 41
  ID    37 | '▁잘' | 예측 횟수: 35
  ID    86 | '▁많이' | 예측 횟수: 31
  ID    48 | '▁것' | 예측 횟수: 31
  ID  6950 | '은' | 예측 횟수: 27
  ID  6943 | '도' | 예측 횟수: 25
  ID    72 | '▁있을' | 예측 횟수: 22
  ID   166 | '▁해보세요' | 예측 횟수: 21
  ID   151 | '▁마세요' | 예측 횟수: 21
  ID  6939 | '을' | 예측 횟수: 19
  ID  6928 | '이' | 예측 횟수: 19
  ID     9 | '▁거' | 예측 횟수: 18
  ID    88 | '▁게' | 예측 횟수: 18


In [29]:
# 답변 길이 분포 확인
# EOS가 압도적인 이유: 답변이 짧아서 대부분의 위치가 PAD/EOS로 채워짐
df['A_len'] = df['A'].apply(lambda x: len(sp.encode_as_ids(x)))
print("답변 토큰 길이 통계:")
print(df['A_len'].describe())
print(f"\n5토큰 이하 답변 비율: {(df['A_len'] <= 5).sum() / len(df) * 100:.1f}%")
print(f"10토큰 이하 답변 비율: {(df['A_len'] <= 10).sum() / len(df) * 100:.1f}%")

# tgt_input 실제 구성 확인 (PAD가 얼마나 많은지)
sample_tgt = next(iter(train_loader))[1]  # 첫 배치의 답변
tgt_input  = sample_tgt[:, :-1]           # 디코더 입력
pad_ratio  = (tgt_input == PAD_ID).float().mean().item()
print(f"\n디코더 입력에서 PAD 비율: {pad_ratio * 100:.1f}%")
print(f"→ 전체 학습 위치의 {pad_ratio*100:.1f}%가 PAD(=EOS 예측 위치)임")

답변 토큰 길이 통계:
count    11823.000000
mean         5.763258
std          2.526971
min          1.000000
25%          4.000000
50%          5.000000
75%          7.000000
max         29.000000
Name: A_len, dtype: float64

5토큰 이하 답변 비율: 54.0%
10토큰 이하 답변 비율: 95.4%

디코더 입력에서 PAD 비율: 79.2%
→ 전체 학습 위치의 79.2%가 PAD(=EOS 예측 위치)임


In [30]:
# 답변 평균 길이가 5.7토큰이므로 MAX_LEN을 20으로 줄임
# 기존 40의 절반 → PAD 비율이 79% → 약 40%로 줄어듦
MAX_LEN    = 20
BATCH_SIZE = 64

# 데이터셋 재구성
train_dataset = ChatDataset(train_df, sp, MAX_LEN)
val_dataset   = ChatDataset(val_df,   sp, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

# PAD 비율 재확인
sample_tgt = next(iter(train_loader))[1]
tgt_input  = sample_tgt[:, :-1]
pad_ratio  = (tgt_input == PAD_ID).float().mean().item()
print(f"조정 후 PAD 비율: {pad_ratio * 100:.1f}%")

# 모델 재초기화 및 학습
model = Transformer(
    vocab_size  = VOCAB_SIZE,
    d_model     = D_MODEL,
    num_heads   = NUM_HEADS,
    d_ff        = D_FF,
    num_layers  = NUM_LAYERS,
    max_len     = MAX_LEN,
    dropout     = DROPOUT
).to(device)

print(f"파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

EPOCHS              = 100
best_val_loss       = float('inf')
early_stop_count    = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = eval_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        early_stop_count = 0
        torch.save(model.state_dict(), '/home/jovyan/work/transformer_chatbot/best_model.pt')
        print(f"          → 모델 저장 (Val Loss: {val_loss:.4f})")
    else:
        early_stop_count += 1
        if early_stop_count >= EARLY_STOP_PATIENCE:
            print(f"\n{EARLY_STOP_PATIENCE} epoch 동안 개선 없음 → 조기 종료")
            break

print(f"\n학습 완료 | Best Val Loss: {best_val_loss:.4f}")

조정 후 PAD 비율: 60.9%
파라미터 수: 8,057,664
Epoch 01 | Train Loss: 5.7606 | Val Loss: 4.9429 | LR: 0.000300
          → 모델 저장 (Val Loss: 4.9429)
Epoch 02 | Train Loss: 4.4250 | Val Loss: 3.6235 | LR: 0.000300
          → 모델 저장 (Val Loss: 3.6235)
Epoch 03 | Train Loss: 3.4320 | Val Loss: 2.6808 | LR: 0.000300
          → 모델 저장 (Val Loss: 2.6808)
Epoch 04 | Train Loss: 2.6820 | Val Loss: 1.9812 | LR: 0.000300
          → 모델 저장 (Val Loss: 1.9812)
Epoch 05 | Train Loss: 2.0727 | Val Loss: 1.4527 | LR: 0.000300
          → 모델 저장 (Val Loss: 1.4527)
Epoch 06 | Train Loss: 1.5776 | Val Loss: 1.0552 | LR: 0.000300
          → 모델 저장 (Val Loss: 1.0552)
Epoch 07 | Train Loss: 1.1719 | Val Loss: 0.7500 | LR: 0.000300
          → 모델 저장 (Val Loss: 0.7500)
Epoch 08 | Train Loss: 0.8485 | Val Loss: 0.5358 | LR: 0.000300
          → 모델 저장 (Val Loss: 0.5358)
Epoch 09 | Train Loss: 0.6021 | Val Loss: 0.3894 | LR: 0.000300
          → 모델 저장 (Val Loss: 0.3894)
Epoch 10 | Train Loss: 0.4219 | Val Loss: 0.2894 | LR:

In [32]:
def predict(sentence, model, sp, max_len=MAX_LEN,  # MAX_LEN이 20으로 업데이트됨
            temperature=0.5,
            repetition_penalty=2.0,
            top_k=10,
            min_len=3):
    model.eval()

    sentence = preprocess(sentence)
    src_ids = [BOS_ID] + sp.encode_as_ids(sentence) + [EOS_ID]
    src_ids += [PAD_ID] * (max_len - len(src_ids))
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    src_mask = create_padding_mask(src)
    with torch.no_grad():
        enc_output = model.encode(src, src_mask)

    tgt_ids = [BOS_ID]
    recent_window = 4

    for step in range(max_len):
        tgt = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        look_ahead_mask = create_look_ahead_mask(tgt)

        with torch.no_grad():
            dec_output = model.decode(tgt, enc_output, look_ahead_mask, src_mask)
            logits = model.fc_out(dec_output[:, -1, :])

            logits[0, PAD_ID] = -1e9
            logits[0, BOS_ID] = -1e9
            logits[0, 1]      = -1e9  # UNK

            # 최소 길이 전까지 EOS 차단
            if step < min_len:
                logits[0, EOS_ID] = -1e9

            for token_id in set(tgt_ids):
                logits[0, token_id] /= repetition_penalty

            for token_id in tgt_ids[-recent_window:]:
                logits[0, token_id] = -1e9

            logits = logits / temperature

            top_k_logits, top_k_ids = torch.topk(logits, top_k, dim=-1)
            probs   = torch.softmax(top_k_logits, dim=-1)
            next_id = top_k_ids[0, torch.multinomial(probs, num_samples=1).item()].item()

        if next_id == EOS_ID:
            break

        tgt_ids.append(next_id)

    return sp.decode_ids(tgt_ids[1:])

# 테스트
tests = [
    "오늘 기분이 너무 안좋아",
    "나 요즘 너무 외로워",
    "밥은 먹었어?",
    "취업이 너무 힘들어",
    "사랑이 뭔지 모르겠어",
]

for t in tests:
    print(f"Q: {t}")
    print(f"A: {predict(t, model, sp)}")
    print()

Q: 오늘 기분이 너무 안좋아
A: 부모님왕 응원합니다 저를 찾아 호칭 따듯한 낭만적인 땐 짜장면헥 파이팅

Q: 나 요즘 너무 외로워
A: 부모님 하아 낭만적인 땐 짜장면 파이팅!!

Q: 밥은 먹었어?
A: 낭만적인 땐 짜장면 부모님 육아

Q: 취업이 너무 힘들어
A: 부모님 낭만적인 땐 짜장면 혼자인 따듯한 불러 가끔 기대가 저를

Q: 사랑이 뭔지 모르겠어
A: 부모님 낭만적인!!

